# bigP3BCI Artifact QC：原始 EEG 初步检查

本 notebook 当前只完成三件事：查看原始 EEG 波形、查看 PSD、记录异常。
不修复数据、不运行 ICA、不删除 epoch、不应用统一滤波，也不覆盖原始 EDF。

当前检查代表性 Subject：`F_05`（Duke 推断组）、`F_03`（ETSU 推断组）、`F_25`（高频表现特殊）。
从本节开始，波形和 PSD 只使用论文中用于 P300 分类的 8 个 EEG 通道。
坏道/坏数据段处理将在下一部分单独进行。

In [ ]:
%matplotlib qt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

PROJECT_ROOT=Path.cwd()
while PROJECT_ROOT!=PROJECT_ROOT.parent and not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    PROJECT_ROOT=PROJECT_ROOT.parent
if not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    raise FileNotFoundError('请在项目目录或子目录启动 Jupyter。')

REPRESENTATIVE_SUBJECTS=['F_05','F_06','F_07','F_08','F_03','F_20','F_21','F_23','F_24','F_25']
SESSION_ID='SE001'
raw_train={}
for subject_id in REPRESENTATIVE_SUBJECTS:
    train_root=PROJECT_ROOT/'data'/'raw'/'bigP3BCI-data'/'StudyF'/subject_id/SESSION_ID/'Train'/'CBCol'
    files=sorted(train_root.glob('*.edf'))
    if not files:
        print('No EDF found:', subject_id, SESSION_ID)
        continue
    path=files[0]
    raw_train[f'{subject_id}_{SESSION_ID}_{path.name}']=mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False)
print('MNE:',mne.__version__,'; representative files:',len(raw_train))
print(list(raw_train))


MNE: 1.12.1 ; representative files: 10
['F_05_SE001_F_05_SE001_CBCol_Train02.edf', 'F_06_SE001_F_06_SE001_CBCol_Train01.edf', 'F_07_SE001_F_07_SE001_CBCol_Train01.edf', 'F_08_SE001_F_08_SE001_CBCol_Train01.edf', 'F_03_SE001_F_03_SE001_CBCol_Train01.edf', 'F_20_SE001_F_20_SE001_CBCol_Train01.edf', 'F_21_SE001_F_21_SE001_CBCol_Train01.edf', 'F_23_SE001_F_23_SE001_CBCol_Train02.edf', 'F_24_SE001_F_24_SE001_CBCol_Train01.edf', 'F_25_SE001_F_25_SE001_CBCol_Train01.edf']


C:\Users\78431\AppData\Local\Temp\ipykernel_18900\1153681929.py:24: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw_train[f'{subject_id}_{SESSION_ID}_{path.name}']=mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_18900\1153681929.py:24: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw_train[f'{subject_id}_{SESSION_ID}_{path.name}']=mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_18900\1153681929.py:24: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw_train[f'{subject_id}_{SESSION_ID}_{path.name}']=mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_18900\1153681929.py:24: RuntimeWarning: Channels contain different lowpass filt

Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none


In [2]:
import re

def assign_bigp3bci_channel_types(raw):
    event_names={
        'StimulusBegin','StimulusType','StimulusCode','CurrentTarget',
        'FakeFeedback','DisplayResults','DisplayResult','SelectedTarget',
        'SelectedRow','SelectedColumn','PhaseInSequence'
    }
    channel_types={}
    for name in raw.ch_names:
        if name in event_names:
            channel_types[name]='stim'
        elif re.match(r'^.+_\d+_\d+$', name):
            channel_types[name]='misc'
        elif name.upper().startswith('EOG'):
            channel_types[name]='eog'
    if channel_types:
        raw.set_channel_types(channel_types, on_unit_change='ignore', verbose=False)
    return raw

PAPER_EEG_CHANNELS=['EEG_Fz','EEG_Cz','EEG_P3','EEG_Pz','EEG_P4','EEG_PO7','EEG_PO8','EEG_Oz']

def get_signal_channels(raw):
    eeg_names=[name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg']
    selected=[name for name in PAPER_EEG_CHANNELS if name in eeg_names]
    missing=[name for name in PAPER_EEG_CHANNELS if name not in eeg_names]
    if missing:
        print('Missing paper channels:', missing)
    return selected

raw_train={
    name: assign_bigp3bci_channel_types(raw)
    for name, raw in raw_train.items()
}
print('Selected paper EEG channels:', get_signal_channels(next(iter(raw_train.values()))))
print('Selected channel count:', len(get_signal_channels(next(iter(raw_train.values())))))

Duke_subjects=['F_05','F_06','F_07','F_08']
ETSU_subjects=['F_03','F_20','F_21','F_23','F_24','F_25']

def show_subject_group(group_name, subject_ids):
    print(f'===== {group_name} =====')
    for subject_id in subject_ids:
        matches=[(filename, raw) for filename, raw in raw_train.items() if filename.startswith(f'{subject_id}_')]
        if not matches:
            print('No loaded file:', subject_id)
            continue
        filename, raw = matches[0]
        eeg=get_signal_channels(raw)
        print(f'{subject_id}: {filename}; channels={len(eeg)}')
        raw.plot(duration=min(60,raw.times[-1]),start=0,picks=eeg,n_channels=len(eeg),remove_dc=False,scalings='auto',title=f'{subject_id}: raw EEG waveform',show=True)
        raw.compute_psd(fmin=0.5,fmax=40,picks=eeg,reject_by_annotation=False,verbose=False).plot(average=True,amplitude=False,show=True)
        raw.compute_psd(fmin=0.5,fmax=min(127,raw.info['sfreq']/2-1),picks=eeg,reject_by_annotation=False,verbose=False).plot(average=True,amplitude=False,show=True)


Selected paper EEG channels: ['EEG_Fz', 'EEG_Cz', 'EEG_P3', 'EEG_Pz', 'EEG_P4', 'EEG_PO7', 'EEG_PO8', 'EEG_Oz']
Selected channel count: 8


## 1. 数据与通道概览

先确认采样率、持续时间、通道类型和已有 bad-channel 标记。事件/反馈通道不参与 EEG 伪迹判断。


In [3]:
overview=[]
for filename,raw in raw_train.items():
    kinds=pd.Series(raw.get_channel_types()).value_counts().to_dict()
    overview.append({'file':filename,'sfreq':raw.info['sfreq'],'duration_sec':raw.times[-1],'n_channels':len(raw.ch_names),'channel_types':kinds,'existing_bads':raw.info['bads']})
display(pd.DataFrame(overview))
for filename,raw in raw_train.items():
    print('\n',filename)
    display(pd.DataFrame({'channel':raw.ch_names,'type':raw.get_channel_types()}))


,file,sfreq,duration_sec,n_channels,channel_types,existing_bads
0,F_05_SE001_F_05_SE001_CBCol_Train02.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
1,F_06_SE001_F_06_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
2,F_07_SE001_F_07_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
3,F_08_SE001_F_08_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
4,F_03_SE001_F_03_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
5,F_20_SE001_F_20_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
6,F_21_SE001_F_21_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
7,F_23_SE001_F_23_SE001_CBCol_Train02.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
8,F_24_SE001_F_24_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]
9,F_25_SE001_F_25_SE001_CBCol_Train01.edf,256.0,217.996094,62,"{'misc': 36, 'eeg': 16, 'stim': 10}",[]



 F_05_SE001_F_05_SE001_CBCol_Train02.edf


,channel,type
0,EEG_F3,eeg
1,EEG_Fz,eeg
2,EEG_F4,eeg
3,EEG_T7,eeg
4,EEG_C3,eeg
...,...,...
57,StimulusBegin,stim
58,StimulusCode,stim
59,CurrentTarget,stim
60,FakeFeedback,stim



 F_06_SE001_F_06_SE001_CBCol_Train01.edf


,channel,type
0,EEG_F3,eeg
1,EEG_Fz,eeg
2,EEG_F4,eeg
3,EEG_T7,eeg
4,EEG_C3,eeg
...,...,...
57,StimulusBegin,stim
58,StimulusCode,stim
59,CurrentTarget,stim
60,FakeFeedback,stim



 F_07_SE001_F_07_SE001_CBCol_Train01.edf


... (11 outputs left out)


 F_24_SE001_F_24_SE001_CBCol_Train01.edf


,channel,type
0,EEG_F3,eeg
1,EEG_Fz,eeg
2,EEG_F4,eeg
3,EEG_T7,eeg
4,EEG_C3,eeg
...,...,...
57,StimulusBegin,stim
58,StimulusCode,stim
59,CurrentTarget,stim
60,FakeFeedback,stim



 F_25_SE001_F_25_SE001_CBCol_Train01.edf


,channel,type
0,EEG_F3,eeg
1,EEG_Fz,eeg
2,EEG_F4,eeg
3,EEG_T7,eeg
4,EEG_C3,eeg
...,...,...
57,StimulusBegin,stim
58,StimulusCode,stim
59,CurrentTarget,stim
60,FakeFeedback,stim


## 1. 查看原始 EEG 波形

使用 `Raw.plot()` 默认查看 60 秒原始 EEG，底部滑块仍可浏览完整记录。重点观察零通道、异常振幅、突然跳变、低频漂移和大幅伪迹。此步骤不滤波、不改变原始数据。


In [4]:
show_subject_group('Duke (inferred)', Duke_subjects)


===== Duke (inferred) =====
F_05: F_05_SE001_F_05_SE001_CBCol_Train02.edf; channels=8


Using matplotlib as 2D backend.
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_06: F_06_SE001_F_06_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_07: F_07_SE001_F_07_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_08: F_08_SE001_F_08_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).


In [5]:
show_subject_group('ETSU (inferred)', ETSU_subjects)

===== ETSU (inferred) =====
F_03: F_03_SE001_F_03_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_20: F_20_SE001_F_20_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_21: F_21_SE001_F_21_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_23: F_23_SE001_F_23_SE001_CBCol_Train02.edf; channels=8


d:\Anaconda\envs\bigp3bci\Lib\site-packages\mne\viz\_mpl_figure.py:2365: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(FigureClass=FigureClass, **kwargs)


Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_24: F_24_SE001_F_24_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_25: F_25_SE001_F_25_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).


## 2. 查看 PSD

先看项目相关的 0.5–40 Hz，再看完整的 0.5–127 Hz。这里只显示频谱，不应用 notch 或其他滤波。


In [15]:
show_subject_group('ETSU (inferred)', ETSU_subjects)


===== ETSU (inferred) =====
F_03: F_03_SE001_F_03_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_20: F_20_SE001_F_20_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_21: F_21_SE001_F_21_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_23: F_23_SE001_F_23_SE001_CBCol_Train02.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_24: F_24_SE001_F_24_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
F_25: F_25_SE001_F_25_SE001_CBCol_Train01.edf; channels=8
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).


## 暂时跳过：EOG/眼动伪迹

当前阶段不运行 EOG detection；后续处理坏道和异常数据段后再决定。


In [7]:
print('EOG detection skipped in this QC stage.')


EOG detection skipped in this QC stage.


## 暂时跳过：ECG/心跳伪迹

当前阶段不运行 ECG detection，也不从 EEG 强行推断 ECG。


In [8]:
print('ECG detection skipped in this QC stage.')


ECG detection skipped in this QC stage.


## 后续步骤：Handling bad channels / bad data spans

本阶段暂不自动计算或标记 bad channels，也不处理异常连续区间。完成原始波形和 PSD 观察后，再进入这一部分。


In [9]:
print('Bad-channel numerical screening deferred to the next stage.')


Bad-channel numerical screening deferred to the next stage.


## 后续步骤：异常连续区间与 annotation

异常连续区间和 `BAD_` annotation 暂不处理，留到下一部分。


In [10]:
print('Bad-span annotation handling deferred to the next stage.')


Bad-span annotation handling deferred to the next stage.


## 当前 notebook 的边界

本 notebook 只做 artifact QC。确认 QC 结果后，下一步才决定是否加入统一滤波、坏道处理、annotation、ICA 或 epoch rejection。跨 session 迁移实验中，所有处理参数都需要记录，并避免使用目标 session 的标签或测试结果泄漏到 source-session 训练流程。
